In [2]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Sample Data (Expanded to allow stratified split with 20% test size)
corpus = [
    "I absolutely loved this product, it works perfectly!",
    "Terrible customer service and the item arrived broken.",
    "Great value for the money, highly recommend.",
    "Worst purchase I have ever made. Do not buy.",
    "It is okay, nothing special but does the job.",
    "Excellent quality, fast shipping!",
    "Horrible experience, will never buy again.",
    "Not bad, but could be better built.",
    "Defective product, completely disappointed.",
    "Awesome product, exceeded my expectations."
]
labels = [1, 0, 1, 0, 1, 1, 0, 1, 0, 1]  # 1 = Positive, 0 = Negative

X_train, X_test, y_train, y_test = train_test_split(
    corpus, labels, test_size=0.2, random_state=42, stratify=labels
)

# 2. Build the Pipeline
# Combining text vectorization and modeling into a clean pipeline prevents data leakage
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),    # Captures both single words and two-word phrases (e.g., "not good")
        max_features=50000,    # Prevents memory blowup on large datasets
        stop_words='english'   # Removes common filler words
    )),
    ('clf', LogisticRegression(
        C=1.0,                 # Inverse regularization strength (tune this!)
        class_weight='balanced', # Crucial if your dataset has class imbalances
        max_iter=1000,
        random_state=42
    ))
])

# 3. Train the Model
text_clf.fit(X_train, y_train)

# 4. Evaluate Performance
predictions = text_clf.predict(X_test)
print(classification_report(y_test, predictions))

# 5. Save the Pipeline for Deployment
joblib.dump(text_clf, 'tfidf_logreg_model.pkl')

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



['tfidf_logreg_model.pkl']